# 03 — Gold publication: eventstream patient-flow tables

| Field | Value |
| ----- | ----- |
| **Sprint** | Sprint 09 v2 — T2.8 |
| **Layer** | `gold/patient-flow/` |
| **Source** | `Tables/silver/eventstream/<eventKind>/` (Delta, validated; silver eventstream stays path-based in P1a) |
| **Target** | `gold.<entity>` (Delta, managed, published, partitioned by `hospitalId`) |
| **Governance** | [ADR-0015](../../../docs/adr/0015-skip-sql-for-mvp-demo.md), [ADR-0016](../../../docs/adr/0016-no-phi-in-mvp-demo-scope.md), [ADR-0013](../../../docs/adr/0013-temporary-us-region-demo-scope.md) |
| **Design spec** | [§4.6 notebook chain](../../../docs/superpowers/specs/2026-07-02-sprint-09-v2-refinement-design.md); [sprint-09 §1.2 governance columns](../../../docs/sprints/sprint-09-master-data-simulation-and-capacity-dashboard.md) |

## Purpose

Publish the 7 validated silver eventstream partitions into **6** patient-flow
gold tables with the full governance column contract (sprint-09 §1.2). The two
encounter kinds (`encounter.admitted` + `encounter.transitioned`) UNION into a
single `gold/patient-flow/encounter/` table; every other kind maps 1:1.

Silver → gold uses `mode('overwrite')` per hop so the notebook is idempotent —
reruns produce the same gold state. Partition-by-`hospitalId` improves downstream
slicer performance in the Power BI capacity dashboard (design spec §6).

## 7 eventKinds → 6 gold tables (design spec §4.6)

| Source eventKind(s) | Gold table |
| ------------------- | ---------- |
| `encounter.admitted` + `encounter.transitioned` (UNION) | `gold/patient-flow/encounter/` |
| `bed.state_changed` | `gold/patient-flow/bed_state/` |
| `bed.assigned` | `gold/patient-flow/bed_assignment/` |
| `forecast.published` | `gold/patient-flow/forecast_output/` |
| `discharge.scored` | `gold/patient-flow/discharge_score/` |
| `discharge.recommended` | `gold/patient-flow/discharge_recommendation/` |

## Mandatory governance columns (sprint-09 §1.2) — identical to `03_gold_master_data.ipynb`

| Column | Value strategy |
| ------ | -------------- |
| `_classification` | Constant `Operational confidential` |
| `_residency_tag` | Preserved from silver (Gate 4 already asserted `{CH-North, US-West}`) |
| `_legal_basis` | Constant `nDSG/KVG` |
| `_retention_class` | Constant `R3` (7 years operational) |
| `_data_quality` | Preserved from silver if present; else default `explicit` |
| `_lineage_ref` | Rewritten to `silver:<eventKind or eventKind_union>:<gold_ts>` |
| `_pseudonymisation_flag` | Constant `false` — no PII in simulator envelopes (ADR-0016 gate already applied at silver) |

In [ ]:
# Parameters (Fabric injects overrides via the papermill-compatible 'parameters' tag).
target_lakehouse = 'lh_ihzhhpf_sit'
run_id = 'run-manual-local'
log_analytics_workspace_id = None                # optional; if None, load summary is only printed
partition_col = 'hospitalId'                      # design spec §4.6 — slicer perf in Power BI dashboard

In [ ]:
# 7 eventKinds → 6 gold tables. Encounter is a UNION of two kinds.
PUBLISH_PLAN = [
    # (gold_entity_name, [source eventKinds])
    ('encounter',                ['encounter.admitted', 'encounter.transitioned']),
    ('bed_state',                ['bed.state_changed']),
    ('bed_assignment',           ['bed.assigned']),
    ('forecast_output',          ['forecast.published']),
    ('discharge_score',          ['discharge.scored']),
    ('discharge_recommendation', ['discharge.recommended']),
]
assert len(PUBLISH_PLAN) == 6, 'design spec §4.6 mandates 6 gold patient-flow tables'
assert sum(len(kinds) for _e, kinds in PUBLISH_PLAN) == 7, 'design spec §4.3 mandates 7 eventKinds'

In [ ]:
from datetime import datetime, timezone
from functools import reduce
from pyspark.sql import DataFrame, functions as F

GOVERNANCE_CONSTANTS = {
    '_classification':        'Operational confidential',
    '_legal_basis':           'nDSG/KVG',
    '_retention_class':       'R3',
    '_pseudonymisation_flag': False,
}

def _now_iso() -> str:
    return datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

def _stamp_governance(df: DataFrame, lineage_label: str, gold_ts: str) -> DataFrame:
    """Apply the 7-column governance contract (sprint-09 §1.2) — identical logic to `03_gold_master_data.ipynb.stamp_governance`."""
    out = df
    for col, val in GOVERNANCE_CONSTANTS.items():
        out = out.withColumn(col, F.lit(val))
    if '_residency_tag' not in out.columns:
        out = out.withColumn('_residency_tag', F.lit('US-West'))
    if '_data_quality' not in out.columns:
        out = out.withColumn('_data_quality', F.lit('explicit'))
    out = out.withColumn('_lineage_ref', F.lit(f'silver:{lineage_label}:{gold_ts}'))
    return out

In [ ]:
# silver eventstream tables remain path-based in P1a (silver notebook not modernized here)
silver_root = 'Tables/silver/eventstream'

def _read_silver(eventKind: str):
    try:
        return spark.read.format('delta').load(f'{silver_root}/{eventKind}')
    except Exception as exc:
        print(f'SKIP [{eventKind}] silver partition not present ({exc})')
        return None

def _union_all(frames):
    frames = [f for f in frames if f is not None]
    if not frames:
        return None
    if len(frames) == 1:
        return frames[0]
    # unionByName + allowMissingColumns handles the case where the two encounter
    # kinds carry slightly different payload structs (admission vs transition sub-record).
    return reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), frames)

# M2 payload flattening (Sprint 10 completion strategy) — Direct Lake measures and
# slicers need flat columns; STRUCT payload isn't queryable via SQL analytics endpoint
# nor as first-class Direct Lake columns. We surface the fields the report bindings
# need. The original `payload` STRUCT stays so nothing else breaks.
FLATTEN_FIELDS = (
    'encounterId',
    'status',
    'previousStatus',
    'admissionType',
    'class',
    'requestedSpecialtyServiceId',
    'expectedLOSDays',
)

def _flatten_payload(df):
    if 'payload' not in df.columns:
        return df
    payload_type = df.schema['payload'].dataType
    is_struct = hasattr(payload_type, 'fieldNames')
    struct_fields = payload_type.fieldNames() if is_struct else []
    # The synthetic batch seed materialises payload as a JSON *string*; the live
    # Eventstream may materialise it as a STRUCT. Support both so a git rebuild and
    # a live run both surface the flat report columns.
    is_json_string = payload_type.typeName() == 'string'
    for f in FLATTEN_FIELDS:
        if is_struct and f in struct_fields:
            df = df.withColumn(f, F.col('payload').getField(f))
        elif is_json_string:
            df = df.withColumn(f, F.get_json_object(F.col('payload'), f'$.{f}'))
        else:
            df = df.withColumn(f, F.lit(None).cast('string'))
    return df

def _add_time_dims(df):
    # M2 heatmap prep: derive simulatedDate (date), simulatedMonth (int 1-12),
    # simulatedWeekday (int 1-7 Mon=1). Enables Month × Weekday heatmap without
    # DAX-side date parsing (Direct Lake calc columns have limited support).
    if 'simulatedAt' not in df.columns:
        return df
    ts = F.to_timestamp(F.col('simulatedAt'))
    return (df
        .withColumn('simulatedDate', F.to_date(ts))
        .withColumn('simulatedMonth', F.month(ts))
        .withColumn('simulatedWeekday', F.dayofweek(ts))  # 1=Sunday..7=Saturday in Spark; adjust in DAX if needed
    )

def publish_entity(gold_entity: str, source_kinds, gold_ts: str):
    frames = [_read_silver(k) for k in source_kinds]
    df = _union_all(frames)
    if df is None:
        return {'gold_entity': gold_entity, 'source_kinds': source_kinds, 'row_count': 0, 'skipped': True}
    lineage_label = source_kinds[0] if len(source_kinds) == 1 else '+'.join(source_kinds)
    out = _stamp_governance(df, lineage_label, gold_ts)
    out = _flatten_payload(out)
    out = _add_time_dims(out)
    writer = out.write.format('delta').mode('overwrite').option('overwriteSchema', 'true')
    if partition_col in out.columns:
        writer = writer.partitionBy(partition_col)
    writer.saveAsTable(f'gold.{gold_entity}')
    total = out.count()
    residency_dist = {r['_residency_tag']: r['count'] for r in out.groupBy('_residency_tag').count().collect()}
    hospital_dist = {r[partition_col]: r['count'] for r in out.groupBy(partition_col).count().collect()} if partition_col in out.columns else {}
    return {
        'gold_entity': gold_entity,
        'source_kinds': source_kinds,
        'row_count': total,
        'residency_distribution': residency_dist,
        'hospital_distribution': hospital_dist,
        'skipped': False,
    }

In [ ]:
# Ensure the gold schema exists (schema-enabled lakehouse managed tables).
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

gold_ts = _now_iso()
results = [publish_entity(entity, kinds, gold_ts) for entity, kinds in PUBLISH_PLAN]

print(f'Gold patient-flow publication summary (run_id={run_id}, gold_ts={gold_ts})')
print('-' * 96)
for r in results:
    tag = 'SKIPPED' if r['skipped'] else 'ok'
    kinds = ','.join(r['source_kinds'])
    print(f"  {r['gold_entity']:<28s} rows={r['row_count']:<6d} sources=[{kinds}] status={tag}")
    if not r['skipped']:
        print(f"    residency={r['residency_distribution']} hospitals={r['hospital_distribution']}")

In [ ]:
# Log Analytics emit — same pattern as `03_gold_master_data.ipynb`. Kept as printed
# payload for local runs; wired to the workspace DCE/DCR when the pipeline injects
# `log_analytics_workspace_id`.
import json as _json

payload = {
    'run_id': run_id,
    'gold_ts': gold_ts,
    'stage': 'gold_patient_flow_publication',
    'gold_tables': results,
}

if log_analytics_workspace_id:
    print(f'LOG_ANALYTICS_EMIT workspace={log_analytics_workspace_id}: {_json.dumps(payload)}')
else:
    print('LOG_ANALYTICS_EMIT (dry-run — no workspace bound):')
    print(_json.dumps(payload, indent=2, default=str))